# 🏋️ Azure AI Search + Microsoft Agent Framework: Fitness-Fun Workshop 🤸

Welcome to this self-guided workshop where you'll:

1. Create an Azure AI Search index containing some sample fitness equipment data.
2. Upload and verify your documents.
3. Create a Microsoft Agent Framework style Foundry agent that grounds its answers in that index.
4. Run a multi-turn conversation to query your data (with a fun fitness twist).

This notebook follows the newer Foundry endpoint-based flow and does not require downgrading `azure-ai-projects`.

Also ensure you've set these environment variables:

- `PROJECT_ENDPOINT`
- `MODEL_DEPLOYMENT_NAME`
- `SEARCH_ENDPOINT`, `SEARCH_API_KEY` (for creating, populating, and querying the index)

Let's get started!

## Prerequisites

Before running the cells below, please verify:

1. You installed the workshop requirements with stable Foundry v2 packages and Microsoft Agent Framework support.
2. Your environment is configured with `PROJECT_ENDPOINT`, `MODEL_DEPLOYMENT_NAME`, `SEARCH_ENDPOINT`, and `SEARCH_API_KEY`.

## 1. Create & Populate Azure AI Search Index

In this section we will:

1. **Create** an Azure AI Search index called `myfitnessindex` with a schema suited for fitness items
2. **Upload** sample documents containing fitness equipment data
3. **Verify** that the documents are searchable

Make sure your environment has the appropriate search credentials (typically obtained via your AI Foundry project).

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.search.documents import SearchClient

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

# Azure AI Search endpoint + admin key (from the Search resource in the Azure portal)
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_credential = AzureKeyCredential(os.environ["SEARCH_API_KEY"])

# Define the index name for our fitness data
index_name = "myfitnessindex"

try:
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
    print("✅ Created SearchIndexClient")

    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )
    print("✅ Created SearchClient for document operations")
except Exception as e:
    print(f"❌ Error creating search clients: {e}")

✅ Created SearchIndexClient
✅ Created SearchClient for document operations


### Define the Index Schema

We will create an index with the following fields:

- `FitnessItemID`: Unique key
- `Name`: Searchable text field (also filterable)
- `Category`: Searchable, filterable, and facetable (e.g. Strength, Cardio, Flexibility)
- `Price`: Numeric field (filterable, sortable, and facetable)
- `Description`: Full-text searchable field

In [2]:
def create_fitness_index():
    fields = [
        SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
        SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
        SearchableField(name="Description", type=SearchFieldDataType.String)
    ]

    index = SearchIndex(name=index_name, fields=fields)

    # Delete the index if it already exists (for a fresh start)
    if index_name in [x.name for x in index_client.list_indexes()]:
        index_client.delete_index(index_name)
        print(f"🗑️ Deleted existing index: {index_name}")

    created = index_client.create_index(index)
    print(f"🎉 Created index: {created.name}")

# Create the index
create_fitness_index()

🎉 Created index: myfitnessindex


### Upload Sample Documents

Now we’ll add some sample fitness items to `myfitnessindex`.

In [3]:
def upload_fitness_docs():
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )

    sample_docs = [
        {
            "FitnessItemID": "1",
            "Name": "Adjustable Dumbbell",
            "Category": "Strength",
            "Price": 59.99,
            "Description": "A compact, adjustable weight for targeted muscle workouts."
        },
        {
            "FitnessItemID": "2",
            "Name": "Yoga Mat",
            "Category": "Flexibility",
            "Price": 25.0,
            "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
        },
        {
            "FitnessItemID": "3",
            "Name": "Treadmill",
            "Category": "Cardio",
            "Price": 499.0,
            "Description": "A sturdy treadmill with adjustable speed and incline settings."
        },
        {
            "FitnessItemID": "4",
            "Name": "Resistance Bands",
            "Category": "Strength",
            "Price": 15.0,
            "Description": "Set of colorful bands for light to moderate resistance workouts."
        }
    ]

    result = search_client.upload_documents(documents=sample_docs)
    print(f"🚀 Upload result: {result}")

upload_fitness_docs()
print("✅ Documents uploaded to search index")

🚀 Upload result: [<azure.search.documents._generated.models._models_py3.IndexingResult object at 0x000002D9C2A5CD70>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x000002D9C2A5CDD0>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x000002D9C2A5CDA0>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x000002D9C2A5CE00>]
✅ Documents uploaded to search index


### Verify the Documents

Let’s perform a basic search query (e.g. for items in the **Strength** category) to ensure everything is working.

In [4]:
results = search_client.search(search_text="Strength", filter=None, top=10)

print("🔍 Search results for 'Strength':")
print("-" * 50)
found_items = False
for doc in results:
    found_items = True
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 50)

if not found_items:
    print("No matching items found.")

🔍 Search results for 'Strength':
--------------------------------------------------
Name: Resistance Bands
Category: Strength
Price: $15.00
Description: Set of colorful bands for light to moderate resistance workouts.
--------------------------------------------------
Name: Adjustable Dumbbell
Category: Strength
Price: $59.99
Description: A compact, adjustable weight for targeted muscle workouts.
--------------------------------------------------


## 2. Create a Foundry Agent that uses Azure AI Search

In this section we use the newer Foundry v2 agent pattern with a Prompt Agent definition to build a fitness shopping assistant. The agent:

- Runs on your Azure OpenAI model (specified by `MODEL_DEPLOYMENT_NAME`).
- Grounds its answers in Azure AI Search: for each question we call a small retrieval function (`retrieve_fitness_items`) that queries `myfitnessindex`, then pass the matching items to the agent as context. This is the retrieval step of Retrieval-Augmented Generation (RAG).
- Holds a multi-turn conversation using the Responses API conversation object.

This keeps the same expected outcome as the previous workshop flow while using the newer Foundry portal-compatible SDK pattern.

In [5]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# New Foundry uses the project endpoint (no connection string).
project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_deployment_name = os.environ["MODEL_DEPLOYMENT_NAME"]

if not model_deployment_name:
    raise ValueError("MODEL_DEPLOYMENT_NAME not set in .env")
if not project_endpoint:
    raise ValueError("PROJECT_ENDPOINT not set in .env")


def retrieve_fitness_items(query: str, k: int = 3) -> str:
    """Retrieve relevant fitness items from Azure AI Search for a user query."""
    hits = search_client.search(search_text=query, top=k)
    lines = [
        f"- {d['Name']} ({d['Category']}, ${d['Price']:.2f}): {d['Description']}"
        for d in hits
    ]
    return "\n".join(lines) if lines else "No matching items found."


project_client = None
agent = None
conversation = None

try:
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()

    # Create a Foundry prompt agent for fitness shopping assistance.
    agent = project_client.agents.create_version(
        agent_name="maf-fitness-shopping-assistant",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
                You are a Fitness Shopping Assistant. Use ONLY the product context provided in each
                message to recommend fitness equipment. Cite the item names you used, and always add a
                short disclaimer that you are not providing medical advice.
            """,
        ),
        description="Fitness shopping assistant grounded with retrieved search context.",
    )

    conversation = openai_client.conversations.create()

    user_queries = [
        "Which items are best for strength training?",
        "I need something for cardio under $300. Any suggestions?",
    ]

    for query in user_queries:
        # RAG step: retrieve relevant items from Azure AI Search, then hand them to the agent.
        context = retrieve_fitness_items(query)
        augmented = f"Product context from the fitness catalog:\n{context}\n\nQuestion: {query}"

        print(f"\n# User: {query}\n")
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=augmented,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"# Agent: {response.output_text}\n")
finally:
    if agent and project_client:
        project_client.agents.delete_version(
            agent_name=agent.name,
            agent_version=agent.version,
        )
        print("🗑️ Cleaned up agent version")


# User: Which items are best for strength training?

# Agent: Best options for strength training from this catalog:

- Adjustable Dumbbell — specifically designed for targeted muscle workouts and the strongest fit for strength training.
- Resistance Bands — good for light to moderate resistance workouts and a versatile strength option.

Less directly focused on strength:
- Yoga Mat — mainly for yoga, Pilates, and floor exercises rather than strength training.

I used: Adjustable Dumbbell, Resistance Bands, Yoga Mat.

Disclaimer: I’m not providing medical advice.


# User: I need something for cardio under $300. Any suggestions?

# Agent: There isn’t a cardio item under $300 in this catalog.

- Treadmill — cardio-focused, but it costs $499.00, so it’s over your budget.
- Resistance Bands — $15.00, but meant for strength training, not cardio.
- Yoga Mat — $25.00, but designed for flexibility work, not cardio.

I used: Treadmill, Resistance Bands, Yoga Mat.

Disclaimer: I’m not providing

## 3. Cleanup

For this demo we already clean up the agent version inside the previous cell. In case you want to remove the search index as well (for a fresh start), run the cell below.

In [6]:
try:
    index_client.delete_index(index_name)
    print(f"🗑️ Deleted index {index_name}")
except Exception as e:
    print(f"Error deleting index: {e}")

🗑️ Deleted index myfitnessindex


# 🎉 Congrats!

You've successfully:

1. Created an Azure AI Search index and populated it with fitness data.
2. Verified the data via a basic search query.
3. Built and ran a Foundry v2 agent flow (Microsoft Agent Framework style) that leverages Azure AI Search to answer natural language queries.

Feel free to explore further enhancements, such as integrating additional tools, evaluation, and tracing workflows in Azure AI Foundry.